# Pre_dict_all

This notebook is to generate the dict that save input `x`, ground truth `y_tr`, and the pred of all models of test dataset.

You need to have trained model save in `logs/{model_name}/`

In [ ]:
import torch

# import sys
# sys.path.append("../")
from utils import Learner

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache() 
# device = "cpu"

save = False

In [ ]:
def save_all_pred(dataloader, pred_model=None, idx=0):
    # batch = next(iter(dataloader))
    # batch_x, batch_y = batch['x'], batch['y']
    
    if pred_model is None:
        xs, ys = [], []
        for i, batch in enumerate(dataloader):
            # Input data (x)
            x = batch['x'].cpu()
        
            # Transformed Ground Truth (y_tr)
            y_tr = batch['y'].cpu()
            xs.append(x)
            ys.append(y_tr)
        return torch.concat(xs), torch.concat(ys)

    else:
        ys = []
        for i, batch in enumerate(dataloader):
            batch_in = batch['x'].to(device)
            y_pred = pred_model(batch_in).detach().cpu()
            ys.append(y_pred)
        return torch.concat(ys)

# Load data

In [ ]:
from utils import MPData
# data_name = 'NS_heat'
# data_name = 'TE_heat'
data_name = 'E_flow'
# data_name = 'MHD'

out_channels= 5 if data_name == 'MHD' else 3
mpdata = MPData(
    "/home/chieh/Data/Multiphysics_Bench/merge_data",
    data_name, 
    dry_run=False,
    batch_size=128
)
train_loader, test_loader = mpdata.loader()

In [ ]:
# pred_dict = torch.load(f'pred_all/pred_{data_name}.pt')
# pred_dict.keys()

In [ ]:
x,y_tr = save_all_pred(test_loader)
pred_dict = {'x':x, 'y_tr':y_tr}
pred_dict, pred_dict.keys()

({'x': tensor([[[[-0.6799, -0.6768, -0.6738,  ..., -0.6129, -0.6157, -0.6185],
            [-0.6772, -0.6741, -0.6710,  ..., -0.6094, -0.6122, -0.6151],
            [-0.6745, -0.6714, -0.6683,  ..., -0.6059, -0.6088, -0.6116],
            ...,
            [-0.6646, -0.6613, -0.6581,  ..., -0.5930, -0.5960, -0.5989],
            [-0.6672, -0.6640, -0.6608,  ..., -0.5964, -0.5994, -0.6023],
            [-0.6699, -0.6667, -0.6635,  ..., -0.5999, -0.6028, -0.6057]]],
  
  
          [[[-0.4622, -0.4571, -0.4520,  ..., -0.3347, -0.3389, -0.3433],
            [-0.4556, -0.4505, -0.4453,  ..., -0.3262, -0.3305, -0.3350],
            [-0.4491, -0.4438, -0.4386,  ..., -0.3177, -0.3221, -0.3266],
            ...,
            [-0.0466, -0.0367, -0.0268,  ...,  0.2020,  0.1937,  0.1852],
            [-0.0503, -0.0404, -0.0305,  ...,  0.1973,  0.1890,  0.1805],
            [-0.0540, -0.0442, -0.0344,  ...,  0.1924,  0.1841,  0.1757]]],
  
  
          [[[-0.7852, -0.7843, -0.7834,  ..., -0.7677, -0

## FNO

In [ ]:
from neuralop.models import FNO
import torch.nn as nn
model = FNO(
        n_modes=(12, 12),
        in_channels=1,
        out_channels=out_channels,
        hidden_channels=32,
        projection_channel_ratio=2
).to(device)
learner = Learner(model, device=device)
learner.load(f'logs/{data_name}/FNO_nop_epoch=500.pt')
pred_dict['FNO'] = save_all_pred(test_loader, model)

Successfully loaded checkpoint from epoch 499


## DeepONet multi-trunk

In [ ]:
from models.DeepONet import MultiTrunkDeepONet
for p in [4096,2048,1024,512,256]:
    model = MultiTrunkDeepONet(
        p=p, out_channels=out_channels).to(device)
    learner = Learner(model, device=device)
    learner.load(f'logs/{data_name}/CDeepONet_p{p}_epoch=500.pt')
    pred_dict[f'CDeepONet({p})'] = save_all_pred(test_loader, model)

Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499


## DeepONet

In [ ]:
from models.DeepONet import DeepONet
import torch.nn as nn
p=256
for p in [4096,2048,1024,512,256]:
    model =  DeepONet(
        p=p, out_channels=out_channels
    ).to(device)
    learner = Learner(model, device=device)
    learner.load(f'logs/{data_name}/MDeepONet_p{p}_epoch=500.pt')
    pred_dict[f'MDeepONet({p})'] = save_all_pred(test_loader, model)

Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499


## CPodONet

In [ ]:
from models.MFPCA import ChannelPodONet
for p in [4096,2048,1024,512,256]:
    model = ChannelPodONet(
        p=p, out_channels=out_channels).to(device)
    learner = Learner(model, device=device)
    # learner.load(f'logs/{data_name}/b32/cPodONet_p{p}_epoch=500.pt')
    learner.load(f'logs/{data_name}/CPodONet_p{p}_epoch=500.pt')

    pred_dict[f'CPodONet({p})'] = save_all_pred(test_loader, model)

Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499


## lrPodONet

In [ ]:
from models.MFPCA import lrPodONet
for p in [4096,2048,1024,512,256]:
    model = lrPodONet(
        p=p, out_channels=out_channels).to(device)
    learner = Learner(model, device=device)
    # learner.load(f'logs/{data_name}/b32/lrPodONet_p{p}_epoch=500.pt')
    learner.load(f'logs/{data_name}/lrPodONet_p{p}_epoch=500.pt')
    pred_dict[f'MPodONet({p})'] = save_all_pred(test_loader, model)

Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499
Successfully loaded checkpoint from epoch 499


# Save Pred_dict

In [ ]:
pred_dict.keys()

dict_keys(['x', 'y_tr', 'FNO', 'CDeepONet(4096)', 'CDeepONet(2048)', 'CDeepONet(1024)', 'CDeepONet(512)', 'CDeepONet(256)', 'MDeepONet(4096)', 'MDeepONet(2048)', 'MDeepONet(1024)', 'MDeepONet(512)', 'MDeepONet(256)', 'CPodONet(4096)', 'CPodONet(2048)', 'CPodONet(1024)', 'CPodONet(512)', 'CPodONet(256)', 'MPodONet(4096)', 'MPodONet(2048)', 'MPodONet(1024)', 'MPodONet(512)', 'MPodONet(256)'])

In [ ]:
# pred_dict_new = {k:pred_dict[k] for k in ['x', 'y_tr', 'FNO', 'DeepONetMP','DeepONet(4096)', 'DeepONet(2048)', 'DeepONet(1024)', 'DeepONet(512)', 'DeepONet(256)', 'CPodONet(8192)', 'CPodONet(4096)', 'CPodONet(2048)', 'CPodONet(1024)', 'CPodONet(512)', 'CPodONet(256)', 'MPodONet(4096)', 'MPodONet(2048)', 'MPodONet(1024)', 'MPodONet(512)', 'MPodONet(256)']}
# pred_dict_new.keys()

In [ ]:
torch.save(pred_dict, f"pred_all/pred_{data_name}.pt")
# torch.save(pred_dict_new, f"pred_all/pred_{data_name}.pt")